In [36]:
BEST_CLUSTER = 4
MAX_DEPTH = 2

In [37]:
# Load cluster

import pandas as pd
import numpy as np

df = pd.read_parquet(
    f"../logs/kmean/clustered_data_gen1_depth{MAX_DEPTH}.parquet"
)

# Outputs rows, columns
df.shape

(158208, 102)

In [38]:
# Quick summary of cluster composition
cluster_summary = df.groupby("cluster").agg(
    rows=("ticker", "count"),
    unique_tickers=("ticker", "nunique"),
    unique_sectors=("sector", "nunique")
)

# Add number of rows per year
year_counts = (
    df.assign(year=df["date"].dt.year)
      .groupby(["cluster", "year"])
      .size()
      .unstack(fill_value=0)
)

# Combine
cluster_summary = cluster_summary.join(year_counts)

cluster_summary.sort_values("rows", ascending=False)

,rows,unique_tickers,unique_sectors,2016,2017,2018,2019,2020,2021
cluster,,,,,,,,,
0,58640,2835,12,1473,187,18387,36164,2394,35
1,38993,2268,12,46,11,5157,7670,26104,5
3,38373,2297,12,0,0,131,619,37623,0
2,22202,2977,12,0,0,2078,39,20085,0


In [39]:
# Temp
df.groupby("cluster")[[
    "future_ret_1y",
    "future_ret_3y",
    "future_ret_5y"
]].median()

,future_ret_1y,future_ret_3y,future_ret_5y
cluster,,,
0,0.031448,0.302176,0.310786
1,0.205108,0.212793,0.349844
2,0.630036,0.553148,0.782391
3,0.442651,0.339438,0.704114


In [40]:
# Look at cluster VS average overall
TARGET_COLS = [
    "future_ret_1d",
    "future_ret_1w",
    "future_ret_1m",
    "future_ret_6m",
    "future_ret_1y",
    "future_ret_3y",
    "future_ret_5y"
]

ID_COLS = ["ticker", "date", "sector"]

PRIM_EXCLUDE = {
    "price",
    "sector_size",
    "rows",
    "sector_size_market",
    "rows_market"
}

SEC_EXCLUDE = {
    "excess_ret_1y", "excess_ret_5y",
    "trend_vs_sector_1y", "trend_vs_sector_5y",
    "drawdown_rel_1y", "drawdown_rel_5y",
    "excess_vs_market_1y", "trend_vs_market_1y",
    "risk_adjusted_1y", "risk_adjusted_5y",
    "quality_score",
    "sector_Unknown"
}

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

numeric_cols.remove("cluster")

feature_cols = [
    c for c in numeric_cols
    if c not in (
        set(ID_COLS)
        | set(TARGET_COLS)
        | PRIM_EXCLUDE
        | SEC_EXCLUDE
    )
]

overall = df[feature_cols].mean()

cluster = df.groupby("cluster")[feature_cols].mean()

for c in cluster.index:
    print("\nCLUSTER", c)

    diff = (
        cluster.loc[c] - overall
    ).abs().sort_values(ascending=False)

    print(diff.head(15))


CLUSTER 0
pe                                       8.087303
sec_ret_5y_dispersion_market             0.188172
sec_avg_ret_3y                           0.163756
sec_avg_ret_3y_market                    0.159907
beta_1y                                  0.146842
sec_strong_drawdown_resilience_market    0.125571
sec_strong_drawdown_resilience           0.120617
ret_3y                                   0.109002
sec_ret_1y_dispersion_market             0.091706
sec_positive_1y_trend_pct_market         0.084598
sec_positive_1y_trend_pct                0.073214
sec_ret_5y_dispersion                    0.073129
sec_breadth_positive_1y_market           0.066453
sec_ret_1y_dispersion                    0.056798
sec_avg_ret_5y                           0.053885
dtype: float64

CLUSTER 1
pe                              4.476624
log_market_cap                  0.496931
ret_5y                          0.423265
ret_3y                          0.315755
ret_1y                          0.275919
beta_1y 

In [41]:
# Compare stock performance

cluster_performance = df.groupby("cluster").agg(
    stocks=("ticker","nunique"),
    rows=("ticker","count"),
    median_1y=("future_ret_1y","median"),
    median_3y=("future_ret_3y","median"),
    median_5y=("future_ret_5y","median"),
    positive_5y=("future_ret_5y", lambda x: (x > 0).mean())
)

cluster_performance.sort_values("median_5y", ascending=False)

,stocks,rows,median_1y,median_3y,median_5y,positive_5y
cluster,,,,,,
2,2977,22202,0.630036,0.553148,0.782391,0.818980
3,2297,38373,0.442651,0.339438,0.704114,0.813593
1,2268,38993,0.205108,0.212793,0.349844,0.713051
0,2835,58640,0.031448,0.302176,0.310786,0.703905


In [42]:
# Top 50 stocks in best cluster
best_cluster = BEST_CLUSTER

best = df[
    df["cluster"] == best_cluster
]

stocks = (
    best.groupby("ticker")
    .agg(
        observations=("ticker","count"),
        avg_future_5y=("future_ret_5y","mean"),
        median_future_5y=("future_ret_5y","median"),
        avg_return_1y=("future_ret_1y","mean"),
        avg_beta=("beta_1y","mean"),
        avg_market_cap=("log_market_cap","mean")
    )
    .sort_values(
        "median_future_5y",
        ascending=False
    )
)

stocks.head(50)

,observations,avg_future_5y,median_future_5y,avg_return_1y,avg_beta,avg_market_cap
ticker,,,,,,


In [43]:
# Cluster stability

cluster_frequency = (
    df.groupby("ticker")["cluster"]
    .agg(
        lambda x: x.value_counts().index[0]
    )
)

cluster_frequency.value_counts()

best_stocks = cluster_frequency[
    cluster_frequency == best_cluster
]

best_stocks.head()

Series([], Name: cluster, dtype: int32)

In [44]:
# Winning cluster VS market

comparison = pd.DataFrame({
    "winning_cluster": df[df.cluster==best_cluster]["future_ret_5y"],
    "all_stocks": df["future_ret_5y"]
})

comparison.describe()

,winning_cluster,all_stocks
count,0.0,158208.000000
mean,NaN,0.824347
std,NaN,1.707217
min,NaN,-0.999578
25%,NaN,-0.000547
50%,NaN,0.469262
75%,NaN,1.161695
max,NaN,56.901147
